# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Each entity is referenced by its `@id` field.

Let's inspect the record sets, fields, and columns available in the metadata.

In [ ]:
# List all available RecordSets and their @id

if hasattr(metadata, 'record_sets'):
    print("Record sets (@id and name):")
    for rs in metadata.record_sets:
        print(f"  @id: {rs['@id']}, name: {rs.get('name', '<none>')}")

        # List fields in this RecordSet
        if 'fields' in rs:
            print("    Fields:")
            for field in rs['fields']:
                print(f"      @id: {field['@id']}, name: {field.get('name', '<none>')}, dataType: {field.get('dataType', '<none>')}")
        # List columns in this RecordSet
        if 'columns' in rs:
            print("    Columns:")
            for col in rs['columns']:
                print(f"      @id: {col['@id']}, name: {col.get('name', '<none>')}, source: {col.get('source', '<none>')}\n")
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the RecordSet and Field `@id`s from the overview above.

_Note: Replace placeholders with the actual `@id`s as needed. For this dataset, we will try to automatically detect record sets, or use an explicit one if available._

In [ ]:
# Gather all record set @ids
record_set_ids = []
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_set_ids = [rs['@id'] for rs in metadata.record_sets]

# If no record sets are listed, try to obtain the ids via metadata.record_set
if not record_set_ids and hasattr(metadata, 'record_set') and metadata.record_set:
    # The Croissant vocabulary sometimes uses 'record_set' (singular though should be a list)
    if isinstance(metadata.record_set, list):
        record_set_ids = [rs['@id'] for rs in metadata.record_set]
    elif isinstance(metadata.record_set, dict):
        record_set_ids = [metadata.record_set['@id']]
    else:
        # In rare case it's just an @id string
        record_set_ids = [metadata.record_set]


print(f"Record set @id list found: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from RecordSet @id: {record_set_id}")
        else:
            print(f"No records found for RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# For demonstration, let's print columns of the first available dataframe
if dataframes:
    primary_rs_id = list(dataframes.keys())[0]
    print(f"Columns for record set {primary_rs_id}:")
    print(dataframes[primary_rs_id].columns.tolist())
    display(dataframes[primary_rs_id].head())
else:
    primary_rs_id = None
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section can include removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis. All references should be by `@id` fields.

In [ ]:
import numpy as np

# Auto-detect a numeric field (by checking dtype or column name), or set manually if needed
df = dataframes.get(primary_rs_id)
if df is not None:
    numeric_field_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]  # Take the first numeric field/column
    else:
        # If none, try to parse as numbers columns with names containing e.g., 'abundance', 'MW', or 'coverage'
        for col in df.columns:
            if any(substr in col.lower() for substr in ['abundance', 'mw', 'coverage', 'count', 'number']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if df[col].notnull().sum() > 0:
                        numeric_field_id = col
                        break
                except Exception:
                    continue
        else:
            numeric_field_id = None

    if numeric_field_id:
        threshold = np.nanmean(df[numeric_field_id])  # Use mean as example filtering threshold

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to select a group-by field, e.g. one with string/object type, not the numeric field
        group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object']
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
            display(grouped_df.head())
        else:
            group_field = None
            print("No suitable group-by field found.")
    else:
        print("Could not detect a numeric field for analysis.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot the distribution of the numeric field and, if possible, boxplot by group_field
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the Mass Spectrometry dataset using its Croissant schema, explored its record sets using the mlcroissant API, extracted records to pandas DataFrames, performed basic filtering and normalization on numeric fields (referenced by their `@id`), grouped data for analysis, and visualized distributions and relationships between fields.

For more advanced exploration and modeling, refer to the dataset's documentation and the mlcroissant [user guide](https://mlcommons.github.io/croissant/) for working with complex schemas and linked data.